In [ ]:
from act_segmentation import VoteExtractor
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.optim as optim
import pandas as pd
from PIL import Image
from tqdm import tqdm
import cv2
import io

In [96]:
extractor = VoteExtractor("src/test2.pdf")
n, t, s = extractor.extract_votes(clean = True)

### DATA PREPROCESSING

In [97]:
splits = {'train': 'data/train-00000-of-00001.parquet', 'validation': 'data/validation-00000-of-00001.parquet', 'test': 'data/test-00000-of-00001.parquet'}
df = pd.read_parquet("hf://datasets/nguyenminh4099/handwritten-digits/" + splits["train"])

In [98]:
def img_to_gray(img_dict):
    return np.array(Image.open(io.BytesIO(img_dict["bytes"])))[:, :, 3]

df["img"] = df["img"].apply(img_to_gray)

In [ ]:
class DigitsDataset(Dataset):
    def __init__(self, df):
        self.x = df["img"].values
        self.y = df["label"].values

    def __len__(self):
        return len(self.x)

    def __getitem__(self, idx):
        img = self.x[idx]
        label = self.y[idx]
        return torch.tensor(img, dtype=torch.float32).unsqueeze(0), torch.tensor(label, dtype=torch.long)
    
dataset = DigitsDataset(df)


### Creation of Model

In [195]:
model_params = {
    "batch_size": 64,
    "learning_rate": 0.01,
    "num_epochs": 2,
}

trainloader = DataLoader(dataset, batch_size=model_params["batch_size"], shuffle=True)

**Model**

In [196]:
class NNModel(nn.Module):
    def __init__(self):
        super(NNModel, self).__init__()
        self.conv1 = nn.Conv2d(in_channels = 1, out_channels = 12, kernel_size = 3, padding = 1)
        self.conv2 = nn.Conv2d(in_channels = 12, out_channels =36, kernel_size = 3, padding = 1)
        self.pool = nn.MaxPool2d(kernel_size = 3, stride = 3)
        self.fc1 = nn.Linear(36*3*3, 128)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(128, 64)
        self.fc3 = nn.Linear(64, 10)

    def forward(self, x):
        x = self.relu(self.conv1(x))
        x = self.pool(x)
        x = self.relu(self.conv2(x))
        x = self.pool(x)
        x = x.view(-1, 36*3*3)
        x = self.relu(self.fc1(x))
        x = nn.functional.dropout(x, p = 0.2, training=self.training)
        x = self.relu(self.fc2(x))
        x = self.fc3(x)
        return x

model = NNModel()

**Training**

In [197]:
optimizer = optim.Adam(model.parameters(), lr=model_params["learning_rate"])
criterion = nn.CrossEntropyLoss()

In [198]:
def fit(model, trainLoader, num_epochs, loss, optimizer):
    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        progress_bar = tqdm(trainLoader, desc=f"Epoch {epoch+1}/{num_epochs}")
        for images, labels in progress_bar:
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
            progress_bar.set_postfix({"Loss": f"{loss.item():.4f}"})
    return model

In [199]:
model = fit(model, trainloader, model_params["num_epochs"], criterion, optimizer)

Epoch 2/2: 100%|██████████| 1347/1347 [00:25<00:00, 51.84it/s, Loss=0.0005]


**Test**

In [200]:
def evaluate(model, testLoader):
    model.eval()
    with torch.no_grad():
        correct = 0
        total = 0
        for images, labels in testLoader:
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    print(f"Accuracy of the model on the test images: {100 * correct / total:.2f}%")

In [204]:
splits = {'train': 'data/train-00000-of-00001.parquet', 'validation': 'data/validation-00000-of-00001.parquet', 'test': 'data/test-00000-of-00001.parquet'}
df = pd.read_parquet("hf://datasets/nguyenminh4099/handwritten-digits/" + splits["test"])

In [205]:
df["img"] = df["img"].apply(img_to_gray)
test_dataset = DigitsDataset(df)
testloader = DataLoader(test_dataset, batch_size=model_params["batch_size"], shuffle=False)

In [206]:
evaluate(model, testloader)

Accuracy of the model on the test images: 99.18%


In [208]:
im1, label1 = test_dataset[0]

In [213]:
output = model(im1)

In [216]:
probability = torch.softmax(output, dim=1)
_, predicted_label = torch.max(probability, 1)
print(f"Predicted label: {predicted_label.item()}, True label: {label1.item()}")

Predicted label: 9, True label: 9


### Test for real images

In [284]:
def max_enclosing_area(contour):
    ct = contour.reshape(-1, 2)
    return np.int32(ct[:,0].max()-ct[:,0].min()) * np.int32(ct[:,1].max()-ct[:,1].min())

def rect_area(contour):
    x,y,w,h = cv2.boundingRect(contour)
    return w*h

def get_digits_contours(image, max_tresh = 0.8, min_tresh = 0.001):
    _, tresh = cv2.threshold(image, 127, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    contours, _ = cv2.findContours(tresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    image_area = image.shape[0] * image.shape[1]
    contours = [contour for contour in contours if rect_area(contour) < image_area * max_tresh and rect_area(contour) > image_area * min_tresh]
    return contours

def draw_digits_contours(image, contours):
    copy_img = image.copy()
    copy_img = cv2.cvtColor(copy_img, cv2.COLOR_GRAY2BGR)
    contour_im1 = cv2.drawContours(copy_img, contours, -1, (255,1,0), thickness =1)
    for contour in contours:
        x,y,w,h = cv2.boundingRect(contour)
        cv2.rectangle(contour_im1, (x,y), (x+w, y+h), (0,255,0), 1)

In [340]:
extractor.display_image(s[2][1])

In [374]:
test_img = s[2][1]
digit_index = 2
digit = get_digits_contours(test_img)[digit_index]
x,y,w,h = cv2.boundingRect(digit)
real_digit = test_img[y:y+h, x:x+w]

In [375]:
digit_tresh = cv2.threshold(real_digit, 127, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)[1]
kernel = np.ones((2,2), np.uint8)
digit_tresh = cv2.morphologyEx(digit_tresh, cv2.MORPH_OPEN, kernel)
digit_modified = cv2.resize(digit_tresh, (28,28), interpolation=cv2.INTER_CUBIC)
digit_tensor = torch.tensor(digit_modified, dtype=torch.float32).unsqueeze(0).unsqueeze(0)

In [376]:
results =  model(digit_tensor)
probability = torch.softmax(results, dim=1)
_, predicted_label = torch.max(probability, 1)
extractor.display_image(digit_modified, prop = 1)
print(f"Predicted label: {predicted_label.item()}")

Predicted label: 8


In [250]:
extractor.display_image(digit_tresh)